# 数值稳定性和模型初始化

## 梯度消失和梯度爆炸

第t层的输出：
$$
f_t(h_{t-1}) = \delta(W^{t}h_{t-1}) \rightarrow \frac{\partial h_t}{\partial h_t} = diag(\delta\prime(W^{t}h^{t-1}))(W^{t})^{T} 
$$

因为激活函数$\delta$是根据元素一一对应计算的，一个值只和当前值相关，不会受到其他值的影响。向量对向量的求导是矩阵，所以只有对角线上的元素非零，其他元素为零。

损失l关于W的梯度：
$$
\frac{\partial l}{\partial W^{t}} = \frac{\partial l}{\partial h_d} \frac{\partial h_d}{\partial h^{d-1}} \dots \frac{\partial h^{t+1}}{\partial h^{t}} \frac{\partial h^{t}}{\partial W^{t}}
$$

当神经网络的层数过多的时候，反向传播时，越到前面的层，梯度累计的相乘就会越多，导致梯度消失或梯度爆炸。  

### 梯度爆炸

生成100个随机高斯矩阵，并将他们与某个初始的随机矩阵相乘，选择尺度$\delta^2$ = 1,矩阵乘积发生爆炸，这种情况是因为深度网络的初始化所导致的。

In [ ]:
import torch
M = torch.normal(0, 1, size=(4,4))
print('一个矩阵 \n',M)
for i in range(100):
    M = torch.mm(M,torch.normal(0, 1, size=(4, 4)))  # torch函数中的矩阵连乘函数

print('乘以100个矩阵后\n', M)

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


一个矩阵 
 tensor([[ 0.4345,  0.3774,  1.6457,  1.3265],
        [ 1.7162,  0.4501,  1.2149,  2.4048],
        [ 0.2058, -0.1643, -0.5886,  0.4144],
        [-1.1584,  1.6158,  0.1120, -0.5214]])
乘以100个矩阵后
 tensor([[ 5.8527e+26,  2.4180e+26,  2.3634e+26, -1.5131e+26],
        [-1.6524e+27, -6.8266e+26, -6.6724e+26,  4.2720e+26],
        [-1.1949e+27, -4.9367e+26, -4.8253e+26,  3.0894e+26],
        [ 6.4220e+27,  2.6532e+27,  2.5933e+27, -1.6603e+27]])


这是因为深度网络的初始化所导致的

## 打破对称性

在神经网络中，“对称性”通常指的是隐藏层中各个神经元的地位完全等价，导致它们在训练过程中始终更新出相同的参数，无法学习到不同的特征。

举例：假设你有一个只包含一个隐藏层（该层有2个隐藏单元）的多层感知机（MLP）。如果你将这个隐藏层的所有权重参数都初始化为相同的常量（例如全部设为0，或者全部设为某个常数 $c$），那么就会发生以下情况：
1. 前向传播完全相同：由于所有隐藏单元的权重完全一样，它们接收到的输入相同，计算出的加权和相同，经过激活函数后输出的值（活性值）也完全相同;
2. 反向传播完全相同：在反向传播计算梯度时，由于前向传播的输出一样，这2个隐藏单元对最终误差的贡献也是相同的。因此，损失函数对这两个单元的权重求导后，计算出的梯度值也是一模一样的;
3. 参数更新后依然相同：当你使用梯度下降（如小批量随机梯度下降）更新参数时，因为它们原本的值一样，减去的梯度也一样，更新后的参数依然是完全相同的。

如果这种情况发生，无论你训练多少次迭代，这个隐藏层中的2个神经元（甚至推广到成百上千个神经元）都会保持相同的参数。这就意味着这多个神经元的行为就像是只有一个神经元一样。网络将永远无法打破这种对称性，因此也就无法发挥多神经元的表达能力去学习数据中复杂且不同的特征。

## 参数初始化
参数初始化可以有效的避免梯度爆炸和消失问题。

### Xavier初始化

在多层网络中，让每一层输出的方差等于输入的方差（前向传播保持尺度不变），同时让流入每一层的梯度方差等于流出每一层的梯度方差（反向传播保持尺度不变）。

在神经网络中，每一层都可以看作是一个线性变换（假设暂时不考虑非线性激活函数，或者在线性区域内）

$x_j$：当前层的第 $j$ 个输入特征。假设输入的均值为 $0$，方差为 $\gamma^2$；

$n_{in}$：当前层的输入特征总数（即输入维度）。

$n_{out}$：当前层的输出特征总数（即输出维度）；

$W_{ij}$：连接第 $j$ 个输入和第 $i$ 个输出的权重参数。我们假设权重是从均值为 $0$、方差为 $\sigma^2$ 的分布中独立随机抽样得来的； 

$o_i$：当前层的第 $i$ 个输出（前向传播的计算结果）。

#### 前向传播

对于前向传播，第 $i$ 个输出的计算公式为：$$o_i = \sum_{j=1}^{n_{in}} W_{ij} x_j$$

为了保持数值稳定性，我们希望输出 $o_i$ 的方差与输入 $x_j$ 的方差（$\gamma^2$）一致。

因为权重 $W_{ij}$ 和输入 $x_j$ 是相互独立的，且它们的均值都为 0，所以 $E[W_{ij} x_j] = E[W_{ij}]E[x_j] = 0$ 。

所以：
$$
E[o_i] = \sum_{j=1}^{n_{in}} E[W_{ij} x_j] = \sum_{j=1}^{n_{in}} E[W_{ij}]E[x_j] = 0
$$

$$
Var[o_i] = E[o_i^2] - E[o_i]^2 = E[o_i^2]
$$

$$Var[o_i] = E\left[ \left( \sum_{j=1}^{n_{in}} W_{ij} x_j \right)^2 \right]$$

由于不同下标的项（如 $W_{ij}x_j$ 和 $W_{ik}x_k$）相互独立且均值为0，交叉项的期望为0。我们只剩下平方项的期望：$$Var[o_i] = \sum_{j=1}^{n_{in}} E[W_{ij}^2 x_j^2]$$

$$E[W_{ij}^2 x_j^2] = E[W_{ij}^2] E[x_j^2] = Var[W_{ij}] Var[x_j]$$

添加求和符号之后

$$Var[o_i] = \sum_{j=1}^{n_{in}} \sigma^2 \gamma^2 = n_{in} \sigma^2 \gamma^2$$

为了让输出的方差 $Var[o_i]$ 等于输入的方差 $\gamma^2$，我们必须设置：$$n_{in} \sigma^2 = 1$$

#### 反向传播

现在考虑反向传播过程 。假设损失函数为 $L$。我们需要将“关于输出的梯度”传递回“关于输入的梯度”:

$\frac{\partial L}{\partial o_i}$：这是从上一层传回来的，关于当前层输出 $o_i$ 的梯度。我们假设其均值为 0，方差为 $\text{Var}[\frac{\partial L}{\partial o}]$;

$\frac{\partial L}{\partial x_j}$：这是我们要求出的，关于当前层输入 $x_j$ 的梯度（它将继续向更浅的层反向传播）。

根据微积分的多元链式法则：$$\frac{\partial L}{\partial x_j} = \sum_{i=1}^{n_{out}} \frac{\partial L}{\partial o_i} \frac{\partial o_i}{\partial x_j}$$

这个公式与前向传播的公式 $o_i = \sum W_{ij} x_j$ 在形式上是完全对称的

* 前向传播是对 $n_{in}$ 个输入求和；
* 反向传播是对 $n_{out}$ 个输出的梯度求和（因为一个输入 $x_j$ 会影响所有 $n_{out}$ 个输出，所以反向传回梯度时要收集所有 $n_{out}$ 个通道的反馈）。

我们假设权重 $W_{ij}$ 与梯度 $\frac{\partial L}{\partial o_i}$ 相互独立。同理，交叉项期望为 0。
计算输入梯度的方差：

$$Var\left[\frac{\partial L}{\partial x_j}\right] = \sum_{i=1}^{n_{out}} E\left[ W_{ij}^2 \left(\frac{\partial L}{\partial o_i}\right)^2 \right]$$

$$Var\left[\frac{\partial L}{\partial x_j}\right] = \sum_{i=1}^{n_{out}} Var[W_{ij}] Var\left[\frac{\partial L}{\partial o_i}\right] = n_{out} \sigma^2 Var\left[\frac{\partial L}{\partial o}\right]$$

除非 $n_{out} \sigma^2 = 1$，否则梯度的方差可能会在层与层之间增大或消失（梯度爆炸或梯度消失） 。因此，为了让反向传播的梯度方差保持不变，我们需要：
  $$n_{out} \sigma^2 = 1$$

#### Xavier初始化方法

$$\frac{1}{2} (n_{in} + n_{out}) \sigma^2 = 1$$